# Topic 08 · Qwen3-TTS: two tokenizers and a streaming speech model: rebuilding the technical report's computations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd08-qwen3-tts-en.ipynb)

This notebook accompanies [Topic 08 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/qwen3-tts/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd08-qwen3-tts.ipynb)): the code,
the data and every check are the same.

Topic 07 tells how MOSS-TTS builds its data and lays out 33 token channels. The
[*Qwen3-TTS*](https://arxiv.org/abs/2601.15621) technical report (Qwen Team, Alibaba, 2026) asks a different
question: how can a model **start speaking right away**, while the text is still arriving? The answer lies in four
places: two tokenizers (25 Hz with one codebook; 12.5 Hz with sixteen layers), a decoder that runs block by block, a
way of laying out text and speech as two parallel streams, and an MTP module that generates the remaining token
layers within the same frame.

We have neither more than 5 million hours of speech nor a 1.7-billion-parameter model. But most of that design is
**arithmetic written out in the report**: how many tokens per second, which block may see which, how long until the
first audio packet comes out. We write each of those computations as code, check it against the report's tables,
then run the parts that can run on real data: the 3,000 spoken English digits of Topic 05, and the stored
measurements of that topic's small tokenizer.

**What you'll do:**
1. Count the two tokenizers: bits per second, the number of steps the main model has to run for each second of
   speech, the number of positions for 10 minutes of continuous speech; then see, in Topic 05's measurements, what
   the first token layer carries.
2. Build the 25 Hz decoder's block-wise attention mask, compute the first audio packet, and catch a place where the
   report contradicts itself.
3. Check the 12 rows of Table 2 (first-packet latency = language-model time + decoding time), then simulate
   packet-by-packet playback.
4. Lay out a sentence of spoken digits as two text–speech streams, following the released code's rules exactly, and
   compare it with the sequential layout when the text arrives gradually from another language model.
5. Measure by counting why the residual layers have to be generated one by one within a frame (MTP), count steps
   against Topic 07's delay pattern, then compare the importance ratios of GRPO and GSPO on a 2,000-token sequence.

**Background:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/),
[Lesson 21 · Diffusion & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/diffusion/),
[Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/),
[Topic 05 · Audio tokenization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-hoa-am-thanh/) and
[Topic 07 · MOSS-TTS](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/moss-tts/).
**Run time:** under a minute on a CPU, including the first data download.

The notebook uses **PyTorch**, like Topics 05 and 07. Every number from the report carries its page number (arXiv
v1). Numbers taken from the released configuration and source code
([QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS)) are marked "released code", because the report does not
give them. Every other number is computed by the cells below.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

In [ ]:
import hashlib
import io
import json
import math
import os
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


def sci(x, digits=1):
    """A very small or very large number: 2·10⁻⁴ (0 prints as 0; no trailing “.0”)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)) + 1e-9)
    m = num(x / 10**e, digits).removesuffix(".0")
    return f"{m}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Play an 8 kHz clip in the notebook (loudness kept, clipped to [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Data: 3,000 spoken digits, and Topic 05's measurements

The [Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, version 1.0.10,
CC BY-SA 4.0 license): 6 speakers, each saying each digit from 0 to 9 fifty times, recorded at 8,000 samples per
second, 16 bits per sample. As in Topics 05 and 07, we check the MD5 of the *contents* of the 3,000 wav files. The
recordings numbered 0 to 4 are the test set (300 recordings), the rest the training set.

Two files from Topic 05, checked by MD5:
- `codec-results.json`: the measurements of that topic's small tokenizer (CAT: 12.5 frames per second, 32 RVQ layers
  × 1,024 codes, 3 seeds) when only the first k layers are kept, k = 1 … 32. Training it takes more than an hour, so
  we read the results instead of rerunning it.
- `codec-weights.json`: here we only use a few real 32-layer token sequences generated by Topic 05's small TTS model.

In [ ]:
SR = 8000
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
WORDS = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


R = fetch_json("codec-results.json", "53c34e984a6054d6135be5afd7c9128d")
W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD is corrupted; download it again"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        waves.append(np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768)
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
dur = np.array([len(x) for x in waves]) / SR
TEST, TRAIN = np.where(idx < 5)[0], np.where(idx >= 5)[0]
print(
    f"{num(len(waves), 0)} recordings, {num(dur.sum() / 60, 1)} minutes; {num(dur.min(), 2)}–{num(dur.max(), 2)} s long"
)
print(
    f"test set {len(TEST)} recordings, training set {num(len(TRAIN), 0)}; Topic 05 measures on {R['split']['test']} recordings"
)
assert len(TEST) == R["split"]["test"] == 300

## 3 · What the report does, and what we rebuild

| Part | Report (page) | Here |
|---|---|---|
| 25 Hz tokenizer | Qwen2-Audio's encoder, with a resampling layer and VQ inserted in the middle; stage 1 learns speech recognition, stage 2 adds a mel decoder (p. 3, Table 3 p. 6) | counting bits (section 4) |
| 25 Hz decoding | a DiT trained with flow matching, in blocks of 8 tokens, block-wise attention mask; then BigVGAN (pp. 3, 6) | the mask, the first packet (section 5) |
| 12 Hz tokenizer | 16 layers × 2,048 codes, layer 0 semantic (WavLM as teacher), 15 RVQ layers, causal ConvNet (pp. 3–4, Table 4 p. 7) | counting bits; what the first layer carries, through Topic 05's measurements (section 4) |
| Latency | Table 2: time to the first packet's tokens + decoding time, at 1, 3 and 6 concurrent requests (pp. 5–6) | checking 12 rows, simulating playback (section 6) |
| Two streams | text and speech concatenated along the channel axis; on receiving a text token, the model immediately predicts the speech tokens (p. 4) | laying out an FSDD sentence by the released code's rules (section 7) |
| MTP | the main model predicts layer 0, the MTP module generates the residual layers (p. 4) | conditional entropy on a small RVQ; counting steps (section 8) |
| Training | pretraining S1–S3 (8,192 → 32,768 tokens); DPO, GSPO, speaker fine-tuning (pp. 4–5) | sequence length (section 4); GRPO and GSPO importance ratios (section 9) |

Table 1 (p. 2) lists ten models. The released code (README, January 2026) has only the 12 Hz tokenizer and the
12 Hz models; the 25 Hz tokenizer and the models that use it have not been released.

In [ ]:
FAMILY = [  # Table 1 (p. 2): (tokenizer, size, variant)
    ("12Hz", 1.7, "Base"), ("12Hz", 1.7, "VoiceDesign"), ("12Hz", 1.7, "CustomVoice"),
    ("12Hz", 0.6, "Base"), ("12Hz", 0.6, "CustomVoice"),
    ("25Hz", 1.7, "Base"), ("25Hz", 1.7, "VoiceEditing"), ("25Hz", 1.7, "CustomVoice"),
    ("25Hz", 0.6, "Base"), ("25Hz", 0.6, "CustomVoice"),
]  # fmt: skip
RELEASED = [m for m in FAMILY if m[0] == "12Hz"]  # released code (README): only the 12 Hz versions
for tok_name in ("12Hz", "25Hz"):
    names = [f"{num(size, 1)} B {kind}" for t, size, kind in FAMILY if t == tok_name]
    print(f"{tok_name}: {', '.join(names)}")
print(f"{len(FAMILY)} models in Table 1, {len(RELEASED)} released")
assert len(FAMILY) == 10 and len(RELEASED) == 5

## 4 · The two tokenizers in numbers

**Qwen-TTS-Tokenizer-25Hz** (p. 3) is built on the audio encoder of Qwen2-Audio, a speech recognition model. Stage 1
continues training it for speech recognition (ASR), after inserting a resampling layer and a vector quantization (VQ)
layer into the middle of the encoder: a single codebook of 32,768 codes, 25 tokens per second (Table 3, p. 6). Stage
2 attaches a convolutional decoder that reconstructs the mel spectrogram from the tokens, then fine-tunes the whole
model, so that the tokens also carry acoustic detail.

**Qwen-TTS-Tokenizer-12Hz** (pp. 3–4) follows the semantic–acoustic split of Mimi (Moshi's tokenizer): 12.5 frames
per second, 16 tokens per frame, 2,048 codes per layer (Table 4, p. 7). The first layer is the semantic layer,
trained to follow WavLM (WavLM as the "teacher"); the 15 layers after it are RVQ, each layer quantizing the residual
that the previous layers have not yet described. The encoder and the decoder are both fully causal, and the decoder
is just a lightweight ConvNet.

These two designs give two very different models. In the 25 Hz version, the main language model generates every
token. In the 12 Hz version, the main model generates only layer 0 of each frame; the other 15 layers are generated by
the MTP module (section 8).

### 🤔 Predict first
The 12 Hz tokenizer uses almost six times as many bits per second as the 25 Hz one. So which version's main model has
to run more steps for each second of speech?

In [ ]:
FPS25, CODES25, LAYERS25 = 25, 32768, 1  # p. 3; Table 3, p. 6
FPS12, CODES12, LAYERS12 = 12.5, 2048, 16  # pp. 3–4; Table 4, p. 7
SR_CODE, HOP_CODE = 24000, 1920  # released code: 24,000 samples per second, 1,920 samples per frame
assert SR_CODE / HOP_CODE == FPS12

TOK = {"25 Hz": (FPS25, LAYERS25, CODES25), "12 Hz": (FPS12, LAYERS12, CODES12)}
INFO = {}
for name, (fps, layers, codes) in TOK.items():
    INFO[name] = {
        "ms per frame": 1000 / fps,
        "bits per token": math.log2(codes),
        "bits per frame": layers * math.log2(codes),
        "bits per second": fps * layers * math.log2(codes),
        "main-model steps per second": fps,  # one step per frame
        "MTP tokens per second": fps * (layers - 1),
        "main-model steps, 10 s sentence": 10 * fps,
    }
print(f"{'':32s} {'25 Hz':>8s} {'12 Hz':>8s}")
for key in INFO["25 Hz"]:
    print(f"{key:32s} " + " ".join(f"{num(INFO[n][key], 1).removesuffix('.0'):>8s}" for n in TOK))
print(f"bits-per-second ratio: {num(INFO['12 Hz']['bits per second'] / INFO['25 Hz']['bits per second'], 2)}")
print(
    f"compare: the MOSS-TTS tokenizer (Topic 07), 12.5 frames × 32 layers × 10 bits = {num(12.5 * 32 * 10, 0)} bits/s"
)
assert INFO["25 Hz"]["bits per second"] == 375 and INFO["12 Hz"]["bits per second"] == 2200
assert (INFO["25 Hz"]["ms per frame"], INFO["12 Hz"]["ms per frame"]) == (40, 80)
assert INFO["12 Hz"]["MTP tokens per second"] == 187.5

Bits per second and steps per second are two different things. The 25 Hz version has fewer bits (375 versus 2,200
bits/s), but its main model has to run 25 steps for each second of speech; the 12 Hz version's main model runs only
12.5 steps, and the remaining 187.5 tokens per second are generated by MTP, a much smaller network. The report argues
that the 12 Hz tokenizer's coarser time step helps the autoregressive model keep long-range dependencies, so it reads
the text more accurately (p. 7). The cell below prints the word error rates (WER) that the report measures for the
four base models, and the results on long texts.

In [ ]:
SEED = {  # Table 5 (p. 8): WER (%) on the Seed-TTS test sets, (test-zh, test-en)
    "25Hz-0.6B": (1.18, 1.64), "25Hz-1.7B": (1.10, 1.49), "12Hz-0.6B": (0.92, 1.32), "12Hz-1.7B": (0.77, 1.24),
}  # fmt: skip
LONG = {"25Hz-1.7B": (1.517, 1.225), "12Hz-1.7B": (2.356, 2.812)}  # Table 10 (p. 11): (long-zh, long-en)
for size in ("0.6B", "1.7B"):
    a, b = SEED[f"25Hz-{size}"], SEED[f"12Hz-{size}"]
    print(f"Seed-TTS, {size}: 25 Hz {num(a[0])} | {num(a[1])} · 12 Hz {num(b[0])} | {num(b[1])}")
    assert b[0] < a[0] and b[1] < a[1], "the 12 Hz version beats the 25 Hz version of the same size on both sets"
a, b = LONG["25Hz-1.7B"], LONG["12Hz-1.7B"]
print(
    f"long texts of over 10 minutes, 1.7B: 25 Hz {num(a[0], 3)} | {num(a[1], 3)} · 12 Hz {num(b[0], 3)} | {num(b[1], 3)}"
)
print(f"(the text on p. 11 gives for the 25 Hz version: {num(1.533, 3)} | {num(1.571, 3)})")
assert a[0] < b[0] and a[1] < b[1], "on long texts, the 25 Hz version wins"

On the Seed-TTS test sets, the 12 Hz version beats the 25 Hz version of the same size on both sets, and the 12 Hz
1.7 B version reaches a WER of 1.24 on test-en, the lowest in Table 5 (pp. 7–8). But on long texts, the 25 Hz version
wins instead; the report suggests that semantic tokens help keep long sequences stable (p. 11). (The text on p. 11
gives the 25 Hz version's WER as 1.533 and 1.571, which differ from Table 10.)

**How many positions for 10 minutes of continuous speech?** The report says the model can read more than 10 minutes
in one go (pp. 2, 11), without splitting the text into chunks like systems that stitch pieces together. The third
pretraining stage (S3) raises the maximum sequence length from 8,192 to 32,768 tokens (p. 5). Each frame is one
position of the main model (in the two-stream layout, section 7, the text shares positions with the speech, so it
costs nothing extra).

In [ ]:
SEQ_S12, SEQ_S3 = 8192, 32768  # maximum sequence length before and during stage S3 (p. 5)
LONG_MIN = 10  # "over 10 minutes" of speech (pp. 2, 11)
for name in TOK:
    fps = INFO[name]["main-model steps per second"]
    print(f"{name}: 10 minutes = {num(LONG_MIN * 60 * fps, 0):>6s} positions · {num(SEQ_S12, 0)} positions = "
          f"{num(SEQ_S12 / fps / 60, 1):>4s} minutes · {num(SEQ_S3, 0)} positions = {num(SEQ_S3 / fps / 60, 1)} minutes")  # fmt: skip
assert LONG_MIN * 60 * FPS12 == 7500 < SEQ_S12 < LONG_MIN * 60 * FPS25 == 15000 < SEQ_S3

For the 25 Hz version, 10 minutes is 15,000 positions, almost twice 8,192: only after S3 does it fit in one
sequence. The 12 Hz version already fits in 8,192 positions (about 10.9 minutes), and 32,768 positions is about
43.7 minutes. That is before counting the lead-in text, the control instructions and the voice prompt, which also
take up positions.

### What the first layer carries: Topic 05's measurements

The report says that the first layer of the 12 Hz tokenizer carries the content and the later layers carry acoustic
detail (pp. 2–3), but it does not measure each layer separately. Topic 05 measures exactly this question on its small
tokenizer: decode from only the first k layers, then let two "judges" listen to the result, a CNN that guesses the
digit (content) and a CNN that guesses the speaker (voice). That tokenizer also has a semantic loss (λ_sem = 20): a
small model reads the tokens and writes out the digit. That is ASR-style semantic supervision, closer to stage 1 of
the 25 Hz tokenizer than to the 12 Hz tokenizer's WavLM teacher. Topic 05 also trains a version without that loss.
The cell below reads both, averaged over 3 seeds on the 300 test recordings.

In [ ]:
KS = np.array(R["ks"])  # number of layers kept: 1, 2, 4, …, 32


def per_layer(name, key):
    """(3 seeds, 9 layer counts): Topic 05's measurements on the 300 test recordings."""
    return np.array([s["test"][key] for s in R["codec"][name]])


ACC = {(n, key): per_layer(n, key) for n in ("ref", "sem0") for key in ("digit", "spk", "mel")}
print("   k |  bit/s |  digit |  voice | no semantic loss: digit | voice")
for i, k in enumerate(KS):
    a = [ACC[n, key][:, i].mean() for n in ("ref", "sem0") for key in ("digit", "spk")]
    print(f"{k:4d} | {num(12.5 * k * 10, 0):>6s} | {pct(a[0]):>6s} | {pct(a[1]):>6s} | {pct(a[2]):>23s} | {pct(a[3])}")
d, s_ = ACC["ref", "digit"].mean(0), ACC["ref", "spk"].mean(0)
i4 = list(KS).index(4)
print(f"\none layer: digit {pct(d[0])} (32 layers: {pct(d[-1])}), voice {pct(s_[0])} (32 layers: {pct(s_[-1])})")
print(
    f"from 1 to 4 layers: digit up {num(100 * (d[i4] - d[0]), 1)} points, voice up {num(100 * (s_[i4] - s_[0]), 1)} points"
)
for key, label in (("digit", "digit"), ("spk", "voice")):
    r1, z1 = ACC["ref", key][:, 0], ACC["sem0", key][:, 0]
    print(f"one layer, {label}: with the semantic loss {pct(r1.min())}–{pct(r1.max())}, "
          f"without {pct(z1.min())}–{pct(z1.max())} (3 seeds)")  # fmt: skip
mel_r, mel_z = ACC["ref", "mel"][:, -1].mean(), ACC["sem0", "mel"][:, -1].mean()
print(f"mel distance at 32 layers: with the semantic loss {num(mel_r)}, without {num(mel_z)} (lower is better)")
print(
    f"judges on the original recordings: digit {pct(R['judges']['digit']['test'])}, voice {pct(R['judges']['spk']['test'])}"
)
assert s_[i4] - s_[0] > 2 * (d[i4] - d[0]), "the voice gains more from the residual layers than the content does"
assert ACC["sem0", "digit"][:, 0].max() < ACC["ref", "digit"][:, 0].min(), "the 3-seed ranges do not overlap"
assert ACC["sem0", "spk"][:, 0].min() < ACC["ref", "spk"][:, 0].max(), "voice at 1 layer: the 3-seed ranges overlap"

The figure below plots the two judges against the number of layers kept (horizontal axis on a log₂ scale): blue
circles are the digit, orange squares the voice; solid lines are the tokenizer with the semantic loss, dashed lines
with hollow markers the one without. The shaded bands span the lowest to the highest seed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
xk = np.log2(KS)
for key, col, mk, label in (("digit", BLUE, "o", "digit"), ("spk", ORANGE, "s", "voice")):
    for n, ls, fill, tag in (("ref", "-", col, "with semantic loss"), ("sem0", "--", "white", "without")):
        v = 100 * ACC[n, key]
        ax.fill_between(xk, v.min(0), v.max(0), color=col, alpha=0.15, lw=0)
        ax.plot(xk, v.mean(0), ls, color=col, marker=mk, mfc=fill, ms=6, label=f"{label} · {tag}")
ax.set_xticks(xk, [str(k) for k in KS])
ax.set(xlabel="token layers kept (k)", ylabel="judge correct (%)", ylim=(65, 95))
ax.legend(fontsize=8, frameon=False, loc="lower right", ncol=2)
fmt_axes(fig)
plt.tight_layout()
plt.show()

One layer is already enough for the judge to hear the right digit 81.9% of the time, close to the 90.3% of all 32
layers; the voice is at only 72.0%. From 1 to 4 layers, the voice gains 18.6 points while the digit gains only 7.0.
Without the semantic loss, the first layer carries clearly less content (71.9%, the 3-seed ranges do not overlap),
while the voice at the first layer changes within the variation between seeds. That is exactly the shape the report
describes: the first layer holds "what is said", the residual layers add "who says it, and how". But the boundary is
not clean: one layer already identifies the voice far better than chance (1/6), and the price of semantic
supervision is slightly worse audio at every level (mel distance at 32 layers 1.59 versus 1.48).

The report's Table 3 is the same trade-off seen from the other side: when stage 2 adds acoustic detail to the 25 Hz
tokens, the recognizer reads the tokens slightly worse in all four columns. Table 4 compares the 12 Hz tokenizer with
Mimi, both 16 layers × 2,048 codes × 12.5 frames/s, on the 2,620 sentences of LibriSpeech test-clean (pp. 6–7).

In [ ]:
ASR_WER = {  # Table 3 (p. 6): WER (%) when recognizing from 25 Hz tokens; C.V. EN, C.V. CN, Fleurs EN, Fleurs CN
    "stage 1": (7.51, 10.73, 3.07, 4.23),
    "stage 2": (10.40, 14.99, 4.14, 4.67),
}
COLS = ("C.V. EN", "C.V. CN", "Fleurs EN", "Fleurs CN")
for c, w1, w2 in zip(COLS, *ASR_WER.values(), strict=True):
    print(f"{c:9s}: stage 1 {num(w1):>5s} → stage 2 {num(w2):>5s}  (+{num(w2 - w1)} points)")
assert all(b > a for a, b in zip(*ASR_WER.values(), strict=True)), "stage 2 raises the WER in every column"

RECON = {  # Table 4 (p. 7): PESQ-WB, PESQ-NB, STOI, UTMOS, SIM (higher is better)
    "Mimi": (2.88, 3.42, 0.94, 3.87, 0.87),
    "Qwen 12 Hz": (3.21, 3.68, 0.96, 4.16, 0.95),
}
METRICS = ("PESQ-WB", "PESQ-NB", "STOI", "UTMOS", "SIM")
print(f"\nMimi → Qwen 12 Hz on {num(2620, 0)} LibriSpeech test-clean sentences:")
print(" · ".join(f"{m} {num(a)} → {num(b)}" for m, a, b in zip(METRICS, *RECON.values(), strict=True)))
assert all(b > a for a, b in zip(*RECON.values(), strict=True))

## 5 · Block-wise 25 Hz decoding

The 25 Hz tokens do not turn into audio by themselves. The decoder has two stages (p. 3): a diffusion Transformer
(DiT), trained with flow matching, turns the tokens into a mel spectrogram; then a modified BigVGAN turns the mel
spectrogram into a waveform. To run in streaming mode, the tokens are grouped into blocks of 8 tokens (p. 6), and the
DiT uses a **block-wise attention mask**: a token in block i may only see blocks i − 3 to i + 1, that is 3 blocks of
look-back, the current block and 1 block of look-ahead (p. 3):

$$M_{ik} = \mathbb{1}\big[\, i-3 \le k \le i+1 \,\big]$$

The look-ahead block improves quality, but at a cost: block 0's mel spectrogram cannot be computed until block 1
exists. BigVGAN also needs 130 ms of extra mel spectrogram after a segment before it can output that segment's
waveform (p. 6).

### 🤔 Predict first
How many tokens must the language model generate before the first audio packet? How many ms of audio does that packet
hold?

In [ ]:
CHUNK, BACK, AHEAD = 8, 3, 1  # tokens per block (p. 6); number of look-back and look-ahead blocks (p. 3)
MS25, VOC_MS = 1000 / FPS25, 130  # 40 ms per token; the trailing mel that BigVGAN needs (p. 6)


def block_mask(n_tok, chunk=CHUNK, back=BACK, ahead=AHEAD):
    """M[a, b] = True when token a may see token b: b's block lies in [block(a) − back, block(a) + ahead]."""
    blk = torch.arange(n_tok) // chunk
    return (blk[None, :] >= blk[:, None] - back) & (blk[None, :] <= blk[:, None] + ahead)


def need_tokens(i, chunk=CHUNK, back=BACK, ahead=AHEAD):
    """Tokens that must exist before the DiT can compute block i's mel spectrogram: the last token block i sees, plus 1."""
    M = block_mask((i + ahead + 2) * chunk, chunk, back, ahead)
    return int(M[i * chunk : (i + 1) * chunk].nonzero()[:, 1].max()) + 1


M64 = block_mask(64)
blk = torch.arange(64) // CHUNK
sees5 = sorted(set(blk[M64[5 * CHUNK]].tolist()))
print(f"mask {tuple(M64.shape)}; block 5 sees blocks {sees5}")
for i in range(4):
    mel_end = (i + 1) * CHUNK * MS25  # mel spectrogram available up to the end of block i
    print(f"block {i}: needs {need_tokens(i):2d} tokens → mel up to {num(mel_end, 0):>5s} ms → "
          f"audio up to {num(mel_end - VOC_MS, 0):>5s} ms")  # fmt: skip
wait, first_mel = need_tokens(0), CHUNK * MS25
print(f"first packet: wait for {wait} tokens, {num(first_mel, 0)} ms of mel spectrogram, {num(first_mel - VOC_MS, 0)} ms "
      f"of audio out; then one {num(first_mel, 0)} ms packet every {CHUNK} tokens")  # fmt: skip
assert (wait, first_mel, first_mel - VOC_MS) == (16, 320, 190), "differs from p. 6 of the report"
assert [need_tokens(i) - need_tokens(i - 1) for i in range(1, 4)] == [CHUNK] * 3

The model has to generate 16 tokens (640 ms of content) before the first packet, and that packet holds only 190 ms of
audio: exactly the numbers on p. 6. After that, every 8 tokens make one 320 ms packet.

**A place where the report contradicts itself.** Page 3 says: "The DiT's receptive field is restricted to 4
blocks—the current block, a 3-block lookback, and a 1-block lookahead." That list is 3 + 1 + 1 = 5 blocks, and the
mask above has block 5 see exactly 5 blocks. The 16 tokens on p. 6 agree with 1 look-ahead block, but say nothing
about the number of look-back blocks: the cell below changes the number of look-back blocks and finds that the number
of tokens to wait for does not change.

In [ ]:
PRINTED_BLOCKS = 4  # "restricted to 4 blocks" (p. 3)
print(f"the list on p. 3: {BACK} + 1 + {AHEAD} = {len(sees5)} blocks; the number on p. 3: {PRINTED_BLOCKS} blocks")
for back in range(4):
    print(f"  look back {back} blocks, look ahead {AHEAD}: a field of {back + 1 + AHEAD} blocks, "
          f"wait for {need_tokens(0, back=back)} tokens")  # fmt: skip
print(f"  look back {BACK} blocks, no look-ahead: wait for {need_tokens(0, ahead=0)} tokens")
assert len(sees5) == BACK + 1 + AHEAD == 5 != PRINTED_BLOCKS
assert {need_tokens(0, back=b) for b in range(4)} == {16} and need_tokens(0, ahead=0) == 8

The figure below is the mask for 64 tokens (8 blocks): a blue cell is an allowed pair (token being computed, token
seen). The orange frame marks the rows of block 5: they see blocks 2 to 6, but not block 7.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.8))
ax.imshow(M64.numpy(), cmap="Blues", vmin=0, vmax=1.6, interpolation="nearest")
for b in range(1, 8):
    ax.axhline(b * CHUNK - 0.5, color="white", lw=1)
    ax.axvline(b * CHUNK - 0.5, color="white", lw=1)
ax.add_patch(Rectangle((-0.5, 5 * CHUNK - 0.5), 64, CHUNK, fill=False, ec=ORANGE, lw=2))
ax.text(63, 5 * CHUNK + 3.5, "block 5", color=ORANGE, ha="right", va="center", fontsize=9, fontweight="bold")
ticks = np.arange(8) * CHUNK + CHUNK / 2 - 0.5
ax.set_xticks(ticks, [str(b) for b in range(8)])
ax.set_yticks(ticks, [str(b) for b in range(8)])
ax.set(xlabel="block of the token seen", ylabel="block of the token being computed")
plt.tight_layout()
plt.show()

**The 12 Hz tokenizer does not wait for future context.** Its decoder looks only at the past, so each token, i.e.
80 ms, can in principle be decoded right away. To avoid the scheduling overhead of packets that are too small, the
report groups 4 tokens into one 320 ms packet (p. 6). The cell below compares the two tokenizers: once the language
model has generated the tokens for so many ms of content, how many ms have become audio.

In [ ]:
PKT12 = 4  # 12 Hz tokens per packet (p. 6)
n25, n12 = np.arange(0, 65), np.arange(0, 33)
ready25 = np.where(n25 >= wait, (n25 // CHUNK - AHEAD) * first_mel - VOC_MS, 0)  # ms of audio decoded
ready12 = (n12 // PKT12) * PKT12 * 80
lag25 = n25 * MS25 - ready25
print(f"25 Hz: tokens for {num(wait * MS25, 0)} ms of content → {num(ready25[wait], 0)} ms of audio; "
      f"the lead is always {num(lag25[wait], 0)} ms at the start of each packet")  # fmt: skip
print(f"12 Hz: tokens for {num(PKT12 * 80, 0)} ms of content → {num(ready12[PKT12], 0)} ms of audio")
assert set(lag25[wait::CHUNK]) == {(1 + AHEAD) * first_mel - first_mel + VOC_MS} == {450}

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot([0, 2560], [0, 2560], ":", color=GREY, lw=1, label="no delay")
ax.step(n25 * MS25, ready25, where="post", color=ORANGE, lw=2, label="25 Hz: block-wise DiT + BigVGAN")
ax.step(n12 * 80, ready12, where="post", color=BLUE, lw=2, ls="--", label="12 Hz: causal ConvNet, 4-token packets")
ax.annotate("", xy=(640, 190), xytext=(640, 640), arrowprops={"arrowstyle": "<->", "color": "k", "lw": 1})
ax.text(670, 400, "450 ms", fontsize=8, va="center")
ax.set(xlabel="content the language model has generated tokens for (ms)", ylabel="audio decoded (ms)")
ax.legend(fontsize=8, frameon=False, loc="upper left")
fmt_axes(fig)
plt.tight_layout()
plt.show()

The orange line (25 Hz) sits 450 ms below the "no delay" line at the start of each packet: 320 ms for the look-ahead
block plus 130 ms for BigVGAN. The blue line (12 Hz) touches the diagonal at each packet. This is latency measured in
*content*: how much the language model has to generate first. How many real milliseconds that becomes depends on how
fast the model generates, and that is Table 2.

## 6 · First-packet latency: Table 2

The report defines (p. 5): **first-packet latency** = the time for the language model to produce enough tokens for
the first packet (LM TTFP) + the time for the tokenizer to decode one packet. The **LM TPP** column is the time for
the language model to produce one packet's tokens in steady state, and **RTF** is compute time divided by audio
duration. Table 2 measures four models at 1, 3 and 6 concurrent requests, on one machine running vLLM. The cell below
copies the table and checks the sum in all 12 rows.

In [ ]:
TABLE2 = {  # Table 2 (p. 5): (concurrency, LM TTFP, decoding, first-packet latency, LM TPP, RTF); times in ms
    "25Hz-1.7B": [(1, 125, 25, 150, 56, 0.253), (3, 222, 62, 284, 64, 0.394), (6, 376, 147, 523, 85, 0.725)],
    "25Hz-0.6B": [(1, 113, 25, 138, 50, 0.234), (3, 198, 62, 260, 59, 0.378), (6, 334, 147, 481, 80, 0.709)],
    "12Hz-1.7B": [(1, 97, 4, 101, 21, 0.313), (3, 190, 5, 195, 24, 0.363), (6, 328, 5, 333, 32, 0.463)],
    "12Hz-0.6B": [(1, 93, 4, 97, 19, 0.288), (3, 174, 5, 179, 22, 0.338), (6, 294, 5, 299, 30, 0.434)],
}
PACKET_MS = 320  # one packet: 8 tokens at 25 Hz or 4 tokens at 12 Hz (p. 6)
print("model      requests  TTFP + decode = first packet  LM TPP   RTF")
for name, rows in TABLE2.items():
    for c, ttfp, dec, fpl, tpp, rtf in rows:
        print(f"{name:10s} {c:6d}     {ttfp:4d} + {dec:3d}    = {fpl:4d} ms  {tpp:5d} ms  {num(rtf, 3)}")
        assert ttfp + dec == fpl, f"{name}, concurrency {c}: TTFP + decoding ≠ first-packet latency"
        assert tpp < PACKET_MS and rtf < 1
print("12 rows: TTFP + decoding = first-packet latency; every LM TPP under 320 ms, every RTF under 1")

The figure below stacks the two parts of the first-packet latency: the bottom part (blue) is the language model's
time, the top part (orange, hatched) the decoding time. Each group is one model; the three bars are 1, 3 and 6
concurrent requests.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for g, (name, rows) in enumerate(TABLE2.items()):
    for j, (c, ttfp, dec, fpl, _, _) in enumerate(rows):
        x = 4 * g + j
        ax.bar(x, ttfp, color=BLUE, width=0.8, label="language model (TTFP)" if x == 0 else None)
        ax.bar(x, dec, bottom=ttfp, color=ORANGE, hatch="//", ec="white", width=0.8,
               label="decoding one packet" if x == 0 else None)  # fmt: skip
        ax.text(x, fpl + 8, str(fpl), ha="center", fontsize=8)
        ax.text(x, -38, f"×{c}", ha="center", fontsize=8, color=GREY)
ax.set_xticks([4 * g + 1 for g in range(4)], list(TABLE2))
ax.tick_params(axis="x", length=0, pad=16)
ax.set(ylabel="first-packet latency (ms)", ylim=(0, 580))
ax.legend(fontsize=8, frameon=False, loc="upper right")
fmt_axes(fig)
plt.tight_layout()
plt.show()

At 1 request, the 12 Hz 0.6 B version delivers its first packet after 97 ms (93 + 4), the number in the report's
abstract. As the number of requests grows, the language model's time grows for all four models, but the decoding part
is very different: the 25 Hz version's DiT and BigVGAN go from 25 to 147 ms, the 12 Hz version's ConvNet only from 4
to 5 ms. The report calls the 12 Hz decoder lightweight and easy to batch (p. 6).

**What unit is RTF measured in?** If RTF is (LM TPP + decoding) divided by the 320 ms of one packet, we can recompute
it. The cell below tries that, and also tries dividing by 80 ms, i.e. one 12 Hz frame.

In [ ]:
print("model      requests  RTF    (TPP+decode)/320  (TPP+decode)/80")
TOL = 0.0005 + 1e-9  # the table rounds to 3 digits
match320, match80, other = [], [], []
for name, rows in TABLE2.items():
    for c, _, dec, _, tpp, rtf in rows:
        a, b = (tpp + dec) / PACKET_MS, (tpp + dec) / 80
        print(f"{name:10s} {c:6d}     {num(rtf, 3)}  {num(a, 4):>16s}  {num(b, 4):>15s}")
        if abs(rtf - a) <= TOL:
            match320.append(name)
        elif abs(rtf - b) <= TOL:
            match80.append(name)
        else:
            other.append((name, c, rtf - b))
print(f"matches /320: {len(match320)} rows ({', '.join(sorted(set(match320)))}); matches /80: {len(match80)} rows")
for name, c, dev in other:
    print(f"no match: {name}, concurrency {c}, off by {num(dev, 4)} from /80")
rtf12 = TABLE2["12Hz-1.7B"][0][5]
print(f"by its RTF, the 12Hz-1.7B version at 1 request spends {num(rtf12 * PACKET_MS, 0)} ms on each 320 ms packet")
assert sorted(set(match320)) == ["25Hz-0.6B", "25Hz-1.7B"] and len(match320) == 6
assert len(match80) == 5 and all(n.startswith("12Hz") for n in match80) and len(other) == 1

The six 25 Hz rows match (TPP + decoding)/320 exactly. Five of the six 12 Hz rows match (TPP + decoding)/80, as if
the 12 Hz version's TPP column were the time for *one frame* rather than for a 4-frame packet; the remaining row
(0.6 B, 6 requests) is off by 0.0035. If so, the 12 Hz version spends about 100 ms on each 320 ms packet at 1
request, and that may be why its RTF (0.313) is higher than the 25 Hz version's (0.253) even though its first-packet
latency is lower. The report does not explain this; it is the notebook's reading, one that fits the numbers, not
something the report says.

**Does playback ever have to wait?** The first packet arrives after the first-packet latency; from then on each
packet takes RTF × 320 ms (the slowest reading: the language model and the decoder run one after the other). The
25 Hz version's first packet holds only 190 ms of audio, later packets 320 ms; the 12 Hz version's packets are always
320 ms. The cell below simulates 40 packets for all 12 rows and adds up the time playback has to pause midway.

In [ ]:
def schedule(arrive, length):
    """Play the packets in order, each once it has arrived and the previous one has finished playing.
    Returns when each packet starts playing, and the total time playback pauses midway."""
    start, stall, end = np.empty(len(arrive)), 0.0, 0.0
    for k, (a, L) in enumerate(zip(arrive, length, strict=True)):
        start[k] = a if k == 0 else max(a, end)
        stall += start[k] - end if k else 0.0
        end = start[k] + L
    return start, stall


def playback(fpl, first_ms, period, n=40, packet=PACKET_MS):
    """Packet 0 arrives at fpl and holds first_ms of audio; packet k ≥ 1 arrives at fpl + k·period and holds 320 ms."""
    arrive = fpl + period * np.arange(n)
    length = np.r_[first_ms, np.full(n - 1, packet)]
    start, stall = schedule(arrive, length)
    return arrive, start, length, stall


STALL = {}
for name, rows in TABLE2.items():
    first = first_mel - VOC_MS if name.startswith("25") else PACKET_MS
    for c, _, dec, fpl, tpp, rtf in rows:
        STALL[name, c] = playback(fpl, first, rtf * PACKET_MS)[3]
    print(f"{name}: pauses " + ", ".join(f"{num(STALL[name, c], 0)} ms at concurrency {c}" for c, *_ in rows))
for name in ("25Hz-1.7B", "25Hz-0.6B"):
    c, _, dec, fpl, tpp, rtf = TABLE2[name][2]
    print(f"{name}, 6 requests: the second packet needs {num(rtf * PACKET_MS, 0)} ms, the first plays only "
          f"{num(first_mel - VOC_MS, 0)} ms; if the stages overlap (the model generates the next packet while the "
          f"previous one is decoded): {max(tpp, dec)} ms per packet, "
          f"pause {num(playback(fpl, first_mel - VOC_MS, max(tpp, dec))[3], 0)} ms")  # fmt: skip
assert {k for k, v in STALL.items() if v > 0} == {("25Hz-1.7B", 6), ("25Hz-0.6B", 6)}
assert round(STALL["25Hz-1.7B", 6]) == 232 - 190

In steady state, playback never has to wait: every RTF is below 1, so each packet is finished before the previous
one has played out. The only place it can pause is right after the 25 Hz version's first packet at 6 requests: the
first packet holds only 190 ms of audio, while the second packet needs 232 ms (the 1.7 B version) if the two stages
run one after the other, so the sound pauses for 42 ms. If they overlap, there is no pause. The report does not say
whether the two stages overlap. The figure below plots two cases at 6 requests: the staircase is the audio received,
the solid line the audio played; where the solid line is flat, playback has paused.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, name in zip(axes, ("25Hz-1.7B", "12Hz-1.7B"), strict=True):
    c, _, dec, fpl, tpp, rtf = TABLE2[name][2]
    first = first_mel - VOC_MS if name.startswith("25") else PACKET_MS
    arrive, start, length, stall = playback(fpl, first, rtf * PACKET_MS, n=8)
    got = np.cumsum(length)
    ax.step(np.r_[0, arrive], np.r_[0, got], where="post", color=ORANGE, lw=1.8, label="audio received")
    tt, played = [0, start[0]], [0, 0]
    for s0, L in zip(start, length, strict=True):
        tt += [s0, s0 + L]
        played += [played[-1], played[-1] + L]
    ax.plot(tt, played, color=BLUE, lw=2, label="audio played")
    if stall > 0:  # the pause: the first packet has finished playing, the second has not arrived yet
        ax.annotate(f"pause {num(stall, 0)} ms", xy=(start[1] - stall / 2, length[0]), xytext=(900, 80),
                    fontsize=8, arrowprops={"arrowstyle": "->", "color": "k", "lw": 0.8})  # fmt: skip
    ax.set(
        title=f"{name}, 6 requests: pause {num(stall, 0)} ms",
        xlabel="time since the request was sent (ms)",
        xlim=(0, 2000),
    )
axes[0].set(ylabel="audio (ms)", ylim=(0, 2200))
axes[0].legend(fontsize=8, frameon=False, loc="upper left")
fmt_axes(fig)
plt.tight_layout()
plt.show()

## 7 · Two streams: text and speech in the same position

The main model is a Qwen3 language model. Text is split by Qwen's standard tokenizer, speech by the Qwen-TTS
tokenizer, and a speaker encoder trained jointly with the main model provides the voice vector (p. 4). The report
describes the layout in one sentence: text and speech are **concatenated along the channel axis**, and on receiving a
text token, the model immediately predicts the corresponding speech tokens (p. 4). Every input follows a chat
template (ChatML), with the control instructions placed at the start (pp. 4–5).

The released code (`modeling_qwen3_tts.py`) gives the details. Each position is the **sum** of two embeddings: one
from the text stream, one from the speech stream. In the default (streaming) mode, the last position of the prefix is
the first text token plus the "start of speech" symbol; from there, each generation step takes the sum of the 16
embeddings of the frame just generated **plus the embedding of the next text token**, then "end of text"
(`tts_eos`), then a padding embedding (`tts_pad`) once the text has run out. The code also has a non-streaming mode
(`non_streaming_mode=True`): the whole text comes first, with the padding symbol in the speech stream there, and only
then the frames. We skip the prefix that both modes share (the role in the chat template, the language tag, the
speaker vector).

Our sentence: george reads the first eight digits of π, separated by 0.1 s of silence (illustration), and each word
is one text token (illustration; Qwen's tokenizer may split differently).

In [ ]:
SENT = (3, 1, 4, 1, 5, 9, 2, 6)
GEORGE, GAP = SPEAKERS.index("george"), 0.1
used, pieces, spans, t0 = {}, [], [], 0.0
for d in SENT:
    k = used.get(d, 0)
    used[d] = k + 1
    c = int(np.where((digit == d) & (spk == GEORGE) & (idx == k))[0][0])
    x = waves[c] * 0.05 / np.sqrt(np.mean(waves[c] ** 2))  # same RMS level
    pieces += [x, np.zeros(int(GAP * SR))]
    spans.append((t0, t0 + len(x) / SR))
    t0 += len(x) / SR + GAP
sent = np.concatenate(pieces[:-1])
TXT = [WORDS[d] for d in SENT]
N_TXT, N_FR = len(TXT), math.ceil(len(sent) / SR * FPS12)


def word_at(f):
    """The word being spoken at the middle of frame f (counting from 0); −1 if that is silence."""
    t = (f + 0.5) / FPS12
    return next((i for i, (a, b) in enumerate(spans) if a <= t < b), -1)


print(f"sentence: {' '.join(TXT)}; {num(len(sent) / SR, 2)} s → {N_FR} frames at 12.5 Hz, {N_TXT} text tokens")
play(sent, "george reads the first eight digits of π")

The cell below lays out the sentence in both modes as a sequence of (text stream, speech stream) pairs, then builds
the actual inputs with small embedding tables (8 dimensions, randomly initialized: an illustration; the 1.7 B model
uses 2,048 dimensions) and 16 layers of random codes in place of real tokens. According to the released
configuration, layer 0's embedding table has 3,072 rows (2,048 codes plus special symbols, among them 2,148 for
padding and 2,149 for "start of speech"), and layers 1–15 have 2,048 rows.

In [ ]:
CODEC_PAD, CODEC_BOS = 2148, 2149  # special symbols of layer 0 (released configuration)
trailing = TXT[1:] + ["<eos>"]  # the text tokens that go with the generation steps: the 2nd word on, then "end of text"
DUAL = [(TXT[0], "<bos>")] + [(trailing[f] if f < len(trailing) else "<pad>", f) for f in range(N_FR)]
SEQ = [(w, "<cpad>") for w in TXT] + [("<eos>", "<cpad>"), ("<pad>", "<bos>")] + [("<pad>", f) for f in range(N_FR)]
print(f"two streams: {len(DUAL)} positions = 1 + {N_FR} frames; sequential: {len(SEQ)} positions = {N_TXT} text + 2 + "
      f"{N_FR} frames; difference {len(SEQ) - len(DUAL)}")  # fmt: skip

torch.manual_seed(0)
DIM = 8
VOCAB = sorted(set(TXT)) + ["<eos>", "<pad>"]
E_text = nn.Embedding(len(VOCAB), DIM)
E_codec = nn.ModuleList([nn.Embedding(3072 if q == 0 else CODES12, DIM) for q in range(LAYERS12)])
print(f"layer 0 embedding table: {num(E_codec[0].num_embeddings, 0)} rows (padding = {num(CODEC_PAD, 0)}, start of "
      f"speech = {num(CODEC_BOS, 0)}); layers 1–15: {num(E_codec[1].num_embeddings, 0)} rows")  # fmt: skip
codes = torch.randint(0, CODES12, (N_FR, LAYERS12), generator=torch.Generator().manual_seed(0))  # illustrative codes


def emb_speech(s):
    if s in ("<bos>", "<cpad>"):
        return E_codec[0](torch.tensor(CODEC_BOS if s == "<bos>" else CODEC_PAD))
    return sum(E_codec[q](codes[s, q]) for q in range(LAYERS12))  # the sum of frame s's 16 embeddings


with torch.no_grad():
    X_dual = torch.stack([E_text(torch.tensor(VOCAB.index(w))) + emb_speech(s) for w, s in DUAL])
    X_seq = torch.stack([E_text(torch.tensor(VOCAB.index(w))) + emb_speech(s) for w, s in SEQ])
    # the same computation, written as in the released code: the frame's sum over layers, plus the next text token or padding
    frames_sum = torch.stack([E_codec[q](codes[:, q]) for q in range(LAYERS12)], 1).sum(1)
    text_next = E_text(torch.tensor([VOCAB.index(trailing[f] if f < len(trailing) else "<pad>") for f in range(N_FR)]))
    assert torch.allclose(X_dual[1:], frames_sum + text_next, atol=1e-5)
print(f"inputs: two streams {tuple(X_dual.shape)}, sequential {tuple(X_seq.shape)}")
assert len(DUAL) == 1 + N_FR and len(SEQ) == N_TXT + 2 + N_FR

In two-stream mode, the output at position 0 is frame 0, and the output at position f is frame f: when it predicts
frame f, the model has read the first f + 1 text tokens (until the text runs out). In sequential mode, frame 0 is
predicted only after the whole text has been read.

### 🤔 Predict first
In two-stream mode, when it predicts frame 5 (seconds 0.40–0.48 of the sentence), how many text tokens has the model
read? Which word is george saying at that moment?

In [ ]:
def read_dual(f):
    """How many text tokens the model has read when it predicts frame f, in two-stream mode."""
    return min(f + 1, N_TXT)


for f in (0, 5, 12, N_FR - 1):
    w = word_at(f)
    said = f"is saying word {w + 1} ({TXT[w]})" if w >= 0 else "is silent"
    print(
        f"frame {f:2d} ({num(f / FPS12, 2)}–{num((f + 1) / FPS12, 2)} s): text tokens read {read_dual(f)}; george {said}"
    )
started = [sum(a <= (f + 0.5) / FPS12 for a, _ in spans) for f in range(N_FR)]  # how many words have started
lead = [read_dual(f) - started[f] for f in range(N_FR)]
f_max = int(np.argmax(lead))
print(f"the text stream leads the speech stream by at most {lead[f_max]} words (at frame {f_max}); the text runs out "
      f"from frame {N_TXT - 1}, the speech has {N_FR - N_TXT} frames to go")  # fmt: skip
assert read_dual(5) == 6 and word_at(5) in (0, -1) and max(lead) >= 5

At frame 5, the model has read 6 text tokens, while george has only got to the first word. The rule "one text token
per step" puts the text stream well ahead of the speech stream, because 12.5 frames per second is much faster than
the rate of spoken words. The model has to learn by itself which word it is reading; according to the report, stage
S1 builds a monotonic mapping from text representations to speech (p. 5).

The figure below draws the two modes by position in the sequence. The "text" rows: a green cell is a text token
(labeled with its digit), E is "end of text", a grey hatched cell is padding. The "speech" rows: B is "start of
speech", a grey hatched cell is padding, each colored cell is one frame, labeled with the digit george is saying in
that frame (blue and orange alternate by word; a white cell with a grey border is silence). The black triangle marks
the position whose output is frame 0.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.1))
rows = {"two streams · text": 0, "two streams · speech": 1, "sequential · text": 2.8, "sequential · speech": 3.8}


def cell(x, y, color, label="", hatch=None, edge="none"):
    ax.add_patch(Rectangle((x, y), 0.92, 0.88, fc=color, ec=edge, lw=0.6 if edge != "none" else 0, hatch=hatch))
    if label:
        ax.text(x + 0.46, y + 0.44, label, ha="center", va="center", fontsize=6, color="white")


for layout, (yt, ys) in ((DUAL, (0, 1)), (SEQ, (2.8, 3.8))):
    for p, (w, s) in enumerate(layout):
        if w in VOCAB[:-2]:
            cell(p, yt, GREEN, str(WORDS.index(w)))
        elif w == "<eos>":
            cell(p, yt, "#333333", "E")
        else:
            cell(p, yt, "#eef0f3", hatch="////", edge=GREY)
        if s == "<bos>":
            cell(p, ys, GREY, "B")
        elif s == "<cpad>":
            cell(p, ys, "#eef0f3", hatch="////", edge=GREY)
        else:
            wi = word_at(s)
            if wi >= 0:
                cell(p, ys, (BLUE, ORANGE)[wi % 2], str(SENT[wi]))
            else:
                cell(p, ys, "white", edge=GREY)
    p0 = 0 if layout is DUAL else N_TXT + 1
    ax.plot(p0 + 0.46, ys + 1.25, "^", color="k", ms=6)
    ax.text(p0 + 1.2, ys + 1.3, "output: frame 0", fontsize=7, va="center")
ax.set_xlim(-0.5, len(SEQ) + 0.5)
ax.set_ylim(5.3, -0.3)
ax.set_yticks([v + 0.44 for v in rows.values()], list(rows))
ax.set_xlabel("position in the sequence (after the shared prefix)")
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
fmt_axes(fig)
plt.tight_layout()
plt.show()

In sequential mode, the sequence is 9 positions longer (8 text tokens and "end of text"; in two-stream mode, the
first text token shares a position with "start of speech"), and frame 0 comes after the whole text. For an
eight-word sentence, that difference is small. It is large when the text **arrives gradually**, for example from a
large language model that is answering: sequential mode has to wait for the answer to finish, two-stream mode only
needs the text to stay ahead of the speech.

The released code takes the whole text at once. But the model is causal, and in two-stream mode frame f depends only
on the first f + 1 text tokens, so the same layout works when the text arrives gradually. The cell below simulates
that under two of the notebook's assumptions: the text arrives at a steady 30 tokens per second (illustration), and
each frame costs RTF × 80 ms of the 12 Hz 1.7 B version at 1 request (the per-frame reading of RTF in section 6).
The first packet is 4 frames (p. 6); packet decoding time is not counted.

In [ ]:
R_TEXT = 30  # text tokens per second from the upstream model (illustration)
STEP = TABLE2["12Hz-1.7B"][0][5] * 80  # ms per frame (inferred from Table 2's RTF)


def frame_times(n_text, n_frames, layout, rate=R_TEXT, step=STEP):
    """When (ms) each frame finishes; text token k arrives at k / rate seconds."""
    arrive = 1000 * np.arange(1, n_text + 1) / rate
    t, out = 0.0, []
    for f in range(n_frames):
        need = arrive[min(f, n_text - 1)] if layout == "dual" else arrive[-1]
        t = max(t, need) + step
        out.append(t)
    return np.array(out)


def first_packet_ms(n_text, layout, **kw):
    return frame_times(n_text, PKT12, layout, **kw)[PKT12 - 1]


print(f"{num(STEP, 1)} ms per frame; text arrives at {R_TEXT} tokens/s")
for n_text in (N_TXT, 50, 200):
    print(f"{n_text:3d} text tokens: first packet after {num(first_packet_ms(n_text, 'dual'), 0):>5s} ms (two streams) · "
          f"{num(first_packet_ms(n_text, 'seq'), 0):>5s} ms (sequential)")  # fmt: skip
assert first_packet_ms(200, "dual") == first_packet_ms(N_TXT, "dual") < first_packet_ms(N_TXT, "seq")

In [ ]:
NS = np.arange(4, 301)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(NS, [first_packet_ms(n, "seq") for n in NS], color=ORANGE, lw=2, label="sequential: wait for all the text")
ax.plot(NS, [first_packet_ms(n, "dual") for n in NS], color=BLUE, lw=2, ls="--", label="two streams")
ax.axvline(N_TXT, color=GREY, ls=":", lw=1)
ax.text(N_TXT + 3, 6000, "our sentence", fontsize=8, color=GREY)
ax.set(xlabel="text tokens in the answer", ylabel="first audio packet after (ms)", yscale="log", ylim=(100, 15000))
ax.legend(fontsize=8, frameon=False, loc="center right")
fmt_axes(fig)
plt.tight_layout()
plt.show()

In two-stream mode, the first packet comes out after the same amount of time however long the answer is: only the
first 4 text tokens are needed. In sequential mode, that time grows with the length of the answer. This is why the
report builds the model for "streaming text input and streaming audio output" (p. 2). Section 10 tries text that
arrives slower than 12.5 tokens per second.

## 8 · MTP: layer 0 first, then the 15 residual layers one by one within the frame

Each 12 Hz frame has 16 tokens. The report splits predicting them into two tiers (p. 4): the main model reads the
combined features of the codebook layers (in section 7: the sum of the previous frame's 16 embeddings, plus a text
token) and predicts layer 0; then the **MTP** (*multi-token prediction*) module generates all the residual layers. In
the released code, MTP is a small Transformer (5 layers, width 1,024, per the released configuration; the main model
has 28 layers, width 2,048 in the 1.7 B version) that takes the main model's hidden state together with the embedding
of layer 0, then **generates autoregressively** exactly `num_code_groups − 1` = 15 tokens, each token seeing the
layers before it in the same frame:

$$p(\mathbf{c}_t \mid \mathbf{h}_t) = p_\theta\big(c_t^{0} \mid \mathbf{h}_t\big) \prod_{j=1}^{15} p_\phi\big(c_t^{j} \mid \mathbf{h}_t,\ c_t^{0}, \dots, c_t^{j-1}\big)$$

Why not predict the 15 residual layers all at once with 15 parallel outputs, to be faster? Because in RVQ, layer j
quantizes the residual left *after* the previous layers have chosen their codes, so layer j depends on those very
codes. We measure that dependence on real data: the 40-band log-mel spectra of FSDD's voiced frames (10 ms frames,
not 80 ms), a small RVQ of 3 layers × 16 codes learned with k-means on the training set, then counting. For each way
of "knowing in advance", we build a frequency table on the training set and compute the cross-entropy (bits per
token) on the test set.

In [ ]:
CFG = {  # released configuration (config.json of Qwen3-TTS-12Hz-1.7B-Base and -0.6B-Base)
    "main model, 1.7 B": {"layers": 28, "width": 2048},
    "main model, 0.6 B": {"layers": 28, "width": 1024},
    "MTP (both sizes)": {"layers": 5, "width": 1024, "tokens per frame": LAYERS12 - 1},
}
for k, v in CFG.items():
    print(f"{k}: " + ", ".join(f"{a} {num(b, 0)}" for a, b in v.items()))


def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Triangular filters on the mel scale, (n_fft/2 + 1, n_mels), as in Topic 05."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


HANN, FB40 = torch.hann_window(256), mel_filters(256, 40)


def logmel(x):
    """(B, L) → (B, 40, frames): log₁₀ magnitude in 40 mel bands; 32 ms window, 10 ms hop."""
    s = torch.stft(x, 256, 80, window=HANN, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ FB40).clamp_min(1e-5).log10().transpose(1, 2)


with torch.no_grad():
    frames, owner, has_prev = [], [], []
    for c, x in enumerate(waves):
        m = ((logmel(torch.tensor(x * 0.1 / np.sqrt(np.mean(x**2)), dtype=torch.float32)[None]) + 5) / 2)[0]
        e = m.mean(0)
        keep = (e > e.max() - 1.5).numpy()  # voiced frames: within 1.5 log₁₀ units of the loudest frame
        frames.append(m[:, keep].T)
        owner.append(np.full(int(keep.sum()), c))
        has_prev.append(np.r_[False, keep[:-1]][keep])  # the 10 ms frame right before is voiced too
    FR = torch.cat(frames)
owner, has_prev = np.concatenate(owner), np.concatenate(has_prev)
is_train = np.isin(owner, TRAIN)
print(
    f"{num(len(FR), 0)} voiced frames × {FR.shape[1]} mel bands; {num(has_prev.sum(), 0)} frames right after a voiced one"
)


def kmeans(X, k, iters, g):
    C = X[torch.randperm(len(X), generator=g)[:k]].clone()
    for _ in range(iters):
        a = torch.cdist(X, C).argmin(1)
        C = torch.stack([X[a == j].mean(0) if (a == j).any() else C[j] for j in range(k)])
    return C


def rvq_codes(X, layers=3, k=16, seed=0):
    """RVQ: each layer is k-means (learned on the training set) of the residual after the previous layers."""
    g = torch.Generator().manual_seed(seed)
    res, out = X.clone(), []
    for _ in range(layers):
        C = kmeans(res[torch.tensor(is_train)], k, 20, g)
        a = torch.cdist(res, C).argmin(1)
        out.append(a)
        res = res - C[a]
    return torch.stack(out, 1).numpy()


def cross_bits(codes_, j, ctx, k=16, alpha=0.5):
    """Cross-entropy (bits) of layer j on the test set given the columns ctx; count table (plus α) from the training set.
    A column c < the number of layers is layer c of this frame; the column "prev" is layer j of the 10 ms frame before."""
    cols = np.c_[codes_, np.r_[codes_[:1, j], codes_[:-1, j]]]  # last column: layer j of the previous frame
    cols = [cols[:, -1] if c == "prev" else cols[:, c] for c in ctx]
    key = sum(col * k**i for i, col in enumerate(cols)) if cols else np.zeros(len(codes_), int)
    trn, tst = is_train & has_prev, ~is_train & has_prev
    T = np.full((k ** len(cols), k), alpha)
    np.add.at(T, (key[trn], codes_[trn, j]), 1)
    T /= T.sum(1, keepdims=True)
    return float(-np.log2(T[key[tst], codes_[tst, j]]).mean())


CODES3 = rvq_codes(FR)
print("RVQ 3 layers × 16 codes; codes of the first 5 frames:", CODES3[:5].tolist())

### 🤔 Predict first
Each layer has 16 codes, i.e. at most 4 bits. Given layer 1 of the frame, does also knowing layer 2 help predict
layer 3: hardly at all, or substantially?

In [ ]:
H = {
    ("layer 2", "nothing known"): cross_bits(CODES3, 1, []),
    ("layer 2", "given layer 1"): cross_bits(CODES3, 1, [0]),
    ("layer 3", "nothing known"): cross_bits(CODES3, 2, []),
    ("layer 3", "given layer 1"): cross_bits(CODES3, 2, [0]),
    ("layer 3", "given layers 1 and 2"): cross_bits(CODES3, 2, [0, 1]),
}
HP = {  # adding layer 3 of the previous frame: part of what the main model's hidden state knows about the past
    "given layer 1": cross_bits(CODES3, 2, ["prev", 0]),
    "given layers 1 and 2": cross_bits(CODES3, 2, ["prev", 0, 1]),
}
for (layer, ctx), v in H.items():
    print(f"{layer}, {ctx:20s}: {num(v, 3)} bits")
gap = H["layer 3", "given layer 1"] - H["layer 3", "given layers 1 and 2"]
gap_prev = HP["given layer 1"] - HP["given layers 1 and 2"]
print(f"also knowing layer 2 saves {num(gap, 3)} bits for layer 3")
print(f"when layer 3 of the previous frame is known too: {num(HP['given layer 1'], 3)} → "
      f"{num(HP['given layers 1 and 2'], 3)} bits (saves {num(gap_prev, 3)})")  # fmt: skip
assert gap > 0.3 and gap_prev > 0.1, "within a frame, layer 3 depends on layer 2 even when layer 1 is known"
assert H["layer 2", "given layer 1"] < H["layer 2", "nothing known"] - 0.15

Substantially. Given layer 1, layer 3 still takes 3.884 bits; given layer 2 as well, only 3.483 bits. A parallel
predictor (all residual layers at once, from the same state) has to choose layer 3 without knowing which code layer
2 chose, so on the test set it spends that many extra bits per frame, for this pair of layers alone. Part of the gap
(a smaller one) remains when layer 3 of the previous frame is known too, i.e. part of the history that the main
model's hidden state has. MTP generates sequentially, so layer j sees every layer before it in the frame. Two
caveats: these are bits, not audio quality; and our RVQ (3 layers × 16 codes on 10 ms frames of a mel spectrogram)
is far smaller than 16 layers × 2,048 codes on 80 ms frames of a waveform.

The figure below plots those cross-entropies: each bar is one way of "knowing in advance", the hatched bars add layer
3 of the previous frame; the dashed line is 4 bits, the level of a code guessed uniformly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
labels = [f"{a}\n{b}" for a, b in H] + [
    "layer 3\ngiven previous frame\n+ layer 1",
    "layer 3\ngiven previous frame\n+ layers 1 and 2",
]
vals = [*H.values(), *HP.values()]
cols = [GREY, BLUE, GREY, BLUE, GREEN, BLUE, GREEN]
for i, (v, col) in enumerate(zip(vals, cols, strict=True)):
    ax.bar(i, v, color=col, hatch="//" if i >= 5 else None, ec="white", width=0.7)
    ax.text(i, v - 0.12, num(v, 2), ha="center", va="top", fontsize=8, color="white", fontweight="bold")
ax.axhline(4, color="k", ls="--", lw=0.8)
ax.text(6.45, 4.05, "4 bits: 16 equally likely codes", ha="right", va="bottom", fontsize=7)
ax.set_xticks(range(len(vals)), labels, fontsize=7)
ax.set(ylabel="bits per token (test set)", ylim=(0, 4.4))
fmt_axes(fig)
plt.tight_layout()
plt.show()

### Counting steps: flat, delay pattern, MTP

Topic 07 compares two approaches for 32 layers: the delay pattern and the Local Transformer. With 16 layers, there
are three:
- **flat**: one main-model step per token, 16 · T steps for T frames;
- **delay pattern** (MOSS-TTS's 8 B main model): layer j shifted by j steps, T + 15 steps;
- **MTP** (Qwen3-TTS 12 Hz, like MOSS-TTS's Local Transformer): T main-model steps and 15 · T small steps.

We use real tokens: 6 digits generated by Topic 05's small TTS model (stored in `codec-weights.json`), taking the
first 16 of the 32 layers.

In [ ]:
SH = W["show"]["0"]
assert SH["k"][0] == 32
A16 = torch.cat([torch.tensor(c)[:, :LAYERS12] for c in SH["codes"][0]])  # (frames, 16 layers)
T6 = len(SH["codes"][0][0])
print(
    f"{len(SH['codes'][0])} digits {SH['target']} → {A16.shape[0]} frames × {A16.shape[1]} layers; the first digit has "
    f"{T6} frames"
)


def delay(A):
    """(T, nq) → (T + nq − 1, nq): layer j shifted by j steps; −1 is an empty cell."""
    T_, nq = A.shape
    D_ = torch.full((T_ + nq - 1, nq), -1)
    for j in range(nq):
        D_[j : j + T_, j] = A[:, j]
    return D_


D16 = delay(A16)
assert torch.equal(torch.stack([D16[j : j + len(A16), j] for j in range(LAYERS12)], 1), A16)


def steps(T, nq=LAYERS12):
    """(main-model steps, small steps, main-model steps until frame 0 has all nq tokens)."""
    return {"flat": (T * nq, 0, nq), "delay": (T + nq - 1, 0, nq), "MTP": (T, T * (nq - 1), 1)}


for T in (T6, len(A16), 75, 125):
    print(f"{T:3d} frames ({num(T / FPS12, 2)} s): " + " · ".join(
        f"{k} {num(a, 0)} large steps" + (f" + {num(b, 0)} small steps" if b else "") for k, (a, b, _) in steps(T).items()))  # fmt: skip
print("large steps until frame 0 has all 16 tokens: " + ", ".join(f"{k} {c}" for k, (_, _, c) in steps(1).items()))
assert D16.shape == (len(A16) + 15, 16) and steps(125)["MTP"] == (125, 1875, 1) and steps(125)["flat"][0] == 2000
assert steps(1)["delay"][2] == 16

The figure below colors each token (layer × frame) of the first digit by the main-model step that generates it, and
writes that step number. In the flat layout and the delay pattern, each layer of frame 0 needs its own large step;
with MTP, all 16 layers of frame t come out at large step t, the 15 residual layers generated one by one by MTP's 15
small steps.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4.2), sharey=True)
order = {
    "flat: 16 · T steps": np.arange(T6)[None, :] * LAYERS12 + np.arange(LAYERS12)[:, None],
    "delay pattern: T + 15 steps": np.arange(T6)[None, :] + np.arange(LAYERS12)[:, None],
    "MTP: T large steps (+ 15 small steps per frame)": np.broadcast_to(np.arange(T6)[None, :], (LAYERS12, T6)),
}
for ax, (title, S) in zip(axes, order.items(), strict=True):
    ax.imshow(S, aspect="auto", cmap="viridis", interpolation="nearest")
    for q in range(LAYERS12):
        for t in range(T6):
            ax.text(t, q, str(S[q, t]), ha="center", va="center", fontsize=6,
                    color="white" if S[q, t] < S.max() * 0.6 else "k")  # fmt: skip
    ax.set_title(title, fontsize=8)
    ax.set_xticks(range(T6), [str(t) for t in range(T6)])
    ax.set_xlabel("frame")
axes[0].set_yticks([0, 5, 10, 15], ["layer 0", "layer 5", "layer 10", "layer 15"])
plt.tight_layout()
plt.show()

For a 10-second sentence (125 frames), the flat layout needs 2,000 large steps; the delay pattern only 140, but frame
0 has all 16 tokens only after 16 steps; MTP needs 125 large steps and 1,875 small ones, and frame 0 has all 16
tokens right after the first large step. That is the report's claim: MTP "enables immediate speech decoding from the
first codec frame" (p. 2). Each small step runs a network much smaller than the main model, but those 15 steps still
run one after another within each frame.

## 9 · Post-training: importance ratios on long sequences

After the three pretraining stages, the report post-trains in three steps (p. 5): **DPO** on preference pairs built
from human feedback on multilingual speech; **GSPO** with rule-based rewards; then lightweight speaker fine-tuning on
the base model. DPO is covered in [Topic 04](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rlhf/) (section 10)
and [Lesson 26](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/). Here we look at what makes GSPO
different.

GRPO (DeepSeekMath, 2024) and GSPO (Zheng et al., 2025, from the same Qwen team) both draw a group of responses from
the old policy $\pi_{\text{old}}$, score each one, then update the new policy $\pi_\theta$. They differ in the
**importance ratio**, the correction for the samples coming from $\pi_{\text{old}}$ rather than $\pi_\theta$:

$$w_t = \frac{\pi_\theta(y_t \mid x, y_{<t})}{\pi_{\text{old}}(y_t \mid x, y_{<t})}\ \ \text{(GRPO: one ratio per token)}, \qquad s = \Big(\frac{\pi_\theta(y \mid x)}{\pi_{\text{old}}(y \mid x)}\Big)^{1/|y|} = \exp\Big(\frac{1}{|y|}\sum_t \log w_t\Big)\ \ \text{(GSPO)}$$

The ratio of the whole response is the product $\prod_t w_t$. GSPO takes the $|y|$-th root of that product (the
geometric mean of the $w_t$) and clips per response instead of per token. The Qwen3-TTS report does not say why it
chose GSPO. The GSPO paper argues that each token's ratio rests on a single sample and so is mostly noise, that this
noise accumulates with the length of the response, and that the unit of optimization should match the unit of the
reward, i.e. the whole response. A spoken sentence is very long when counted in tokens: 10 seconds of 12 Hz speech is
125 frames, i.e. 2,000 tokens counting all 16 layers (the report does not say which layers GSPO is computed on).

A toy experiment: each position has a distribution over 2,048 codes (random logits), the new policy is the old one
plus a little noise in the logits (as after one update step), and the positions are independent of each other.
Everything here is an illustration. We draw 400 sequences of 2,000 tokens from the old policy, then measure three
kinds of ratio as a function of the length L.

In [ ]:
EPS_GRPO = (0.2, 0.27)  # GRPO's left/right clipping range in the GSPO paper's experiments (section 5.1)
LMAX, NSEQ = 2000, 400


def toy_ratios(sigma, seed=0):
    """log w_t of NSEQ sequences of length LMAX drawn from the old policy; new policy = old + noise σ in the logits."""
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(LMAX, CODES12, generator=g) * 2
    lp_old = torch.log_softmax(z, -1)
    lp_new = torch.log_softmax(z + sigma * torch.randn(LMAX, CODES12, generator=g), -1)
    y = torch.multinomial(lp_old.exp(), NSEQ, replacement=True, generator=g)  # (position, sequence)
    kl = float((lp_old.exp() * (lp_old - lp_new)).sum(-1).mean())
    return (lp_new.gather(1, y) - lp_old.gather(1, y)).T.numpy(), kl


def short(v):
    return sci(v) if v < 0.01 else num(v, 3)


def bands(logw, L):
    """The 5%, 50%, 95% percentiles of the product of the ratios and of the GSPO ratio, and the share of GRPO tokens
    outside the clipping range."""
    lp = logw[:, :L].sum(1)
    prod, gspo = np.exp(np.percentile(lp, [5, 50, 95])), np.exp(np.percentile(lp / L, [5, 50, 95]))
    w = np.exp(logw[:, :L])
    out = float(((w < 1 - EPS_GRPO[0]) | (w > 1 + EPS_GRPO[1])).mean())
    return prod, gspo, np.exp(np.percentile(logw[:, :L], [5, 95])), out


LOGW, KL = toy_ratios(0.1)
print(f"mean KL per token {sci(KL)} nat; standard deviation of log w_t: {num(LOGW.std(), 3)}")
print(f"GRPO's clipping range: [{num(1 - EPS_GRPO[0], 2)}, {num(1 + EPS_GRPO[1], 2)}]; every range below is 5%–95%")
for L in (1, 16, 125, 2000):
    prod, gspo, tok, out = bands(LOGW, L)
    print(f"L = {num(L, 0):>5s}: product ∏w_t {short(prod[0])} … {short(prod[2])} (median {short(prod[1])}) · "
          f"GSPO s {num(gspo[0], 4)} … {num(gspo[2], 4)} · per token {num(tok[0], 3)} … {num(tok[1], 3)}, "
          f"outside the clipping range {pct(out)}")  # fmt: skip
p16, g16, _, o16 = bands(LOGW, 16)
p2k, g2k, _, o2k = bands(LOGW, 2000)
print(f"L = 2,000: the product spans {num(math.log10(p2k[2] / p2k[0]), 1)} orders of magnitude; GRPO clips about "
      f"{round(o2k * 2000)} tokens per sequence")  # fmt: skip
assert p2k[2] / p2k[0] > 1e5 > p16[2] / p16[0] and g2k[2] - g2k[0] < 0.01 < g16[2] - g16[0]
assert 0.01 < o16 < 0.04 and 0.01 < o2k < 0.04, "the share of clipped tokens barely changes with length"

The figure below plots the range from the 5th to the 95th percentile against sequence length (horizontal axis on a
log scale). Left (vertical axis also on a log scale): the product of all the token ratios. Right (ordinary vertical
axis): the GSPO ratio (blue) and the per-token ratio that GRPO uses (green, hatched), with the two lines of GRPO's
clipping range. The dotted lines are 125 tokens (only layer 0 of a 10-second sentence) and 2,000 tokens (all 16
layers).

In [ ]:
LS = np.unique(np.round(np.logspace(0, np.log10(LMAX), 40)).astype(int))
B = [bands(LOGW, L) for L in LS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.fill_between(LS, [b[0][0] for b in B], [b[0][2] for b in B], color=ORANGE, alpha=0.4, lw=0, label="product ∏ w_t")
a1.plot(LS, [b[0][1] for b in B], color=ORANGE, lw=1.5, label="median")
a1.set(yscale="log", ylim=(1e-10, 10), ylabel="importance ratio", title="product of all token ratios")
a2.fill_between(LS, [b[2][0] for b in B], [b[2][1] for b in B], color=GREEN, alpha=0.3, hatch="//", lw=0,
                label="GRPO: per-token ratio")  # fmt: skip
a2.fill_between(LS, [b[1][0] for b in B], [b[1][2] for b in B], color=BLUE, alpha=0.6, lw=0, label="GSPO: s")
for v in (1 - EPS_GRPO[0], 1 + EPS_GRPO[1]):
    a2.axhline(v, color=GREEN, ls="--", lw=0.8)
a2.text(1.1, 1 + EPS_GRPO[1] + 0.005, "GRPO's clipping range", fontsize=7, color=GREEN, va="bottom")
a2.set(ylim=(0.6, 1.33), title="GSPO and GRPO")
for ax in (a1, a2):
    ax.set(xscale="log", xlabel="sequence length L (tokens)")
    ax.axhline(1, color="k", lw=0.6)
    for L, txt in ((125, "125"), (2000, "2,000")):
        ax.axvline(L, color=GREY, ls=":", lw=1)
        ax.text(L * 0.93, ax.get_ylim()[1], txt, ha="right", va="top", fontsize=8, color=GREY)
    ax.legend(fontsize=8, frameon=False, loc="lower left")
fmt_axes(fig)
plt.tight_layout()
plt.show()

At L = 2,000, the product of the ratios spans 6.4 orders of magnitude between the 5th and 95th percentiles, and its
median is tiny: used as a weight, a few rare sequences would carry nearly all of it. The GSPO ratio, on the other
hand, narrows as the sequence grows longer, because the mean of many log w_t is less noisy than each one. The
per-token ratio that GRPO uses has the same spread at every length, and an almost constant share of tokens (about
2%) falls outside the clipping range: with 2,000 tokens that is a few dozen tokens per sequence, and which tokens get
clipped is down to noise, not to whether the whole response is good or bad. Because s is so narrow, the GSPO paper
(section 5.1) gives it a clipping range of the order of 3·10⁻⁴ and 4·10⁻⁴, hundreds of times smaller than GRPO's.
This toy only shows how the spread of the ratios depends on length; it does not show that GSPO trains a better TTS
model.

## 10 · Things to try

### The 25 Hz decoder's blocks
A smaller block brings the first packet earlier, but BigVGAN still needs 130 ms of trailing mel spectrogram. If a
block is shorter than 130 ms, the first packet has to wait for another block. Change `CHUNKS` or `AHEADS` and rerun.

In [ ]:
CHUNKS, AHEADS = (2, 4, 8, 16), (0, 1, 2)


def first_audio(chunk, ahead):
    """(tokens to wait for, ms of audio in the first packet): the first packet is the first mel block i past 130 ms."""
    i = next(i for i in range(99) if (i + 1) * chunk * MS25 > VOC_MS)
    return need_tokens(i, chunk=chunk, ahead=ahead), (i + 1) * chunk * MS25 - VOC_MS


for chunk in CHUNKS:
    cells = [first_audio(chunk, a) for a in AHEADS]
    print(f"block of {chunk:2d} tokens ({num(chunk * MS25, 0):>3s} ms): " + " · ".join(
        f"look-ahead {a}: wait {w_:2d} tokens, first packet {num(au, 0):>3s} ms" for a, (w_, au) in zip(AHEADS, cells, strict=True)))  # fmt: skip
assert first_audio(8, 1) == (16, 190) and first_audio(4, 1) == (8, 30)

A 4-token block makes the model wait for only 8 tokens, but the first packet holds only 30 ms of audio; a 2-token
block (80 ms) is even shorter than what BigVGAN holds back, so the first packet has to wait for the second block. A
16-token block gives a 510 ms first packet but makes the model wait for 32 tokens. Each extra look-ahead block makes
the model wait one more block of tokens, without changing the amount of audio in the first packet.

### Slow text
In two-stream mode, each frame needs a new text token until the text runs out, i.e. 12.5 tokens per second of
speech. If the upstream model writes slower than that, the speech has to pause and wait for the text. The cell below
uses a 200-token answer, each token matching the average number of speech frames per token in our sentence
(illustration), and counts the time spent paused after playback has started.

In [ ]:
N_LONG = 200
FR_PER_TOK = round(N_FR / N_TXT)
print(f"{N_LONG} text tokens × {FR_PER_TOK} frames per token (our sentence: {N_FR} frames for {N_TXT} tokens)")
for rate in (5, 10, 12.5, 30, 100):
    ft = frame_times(N_LONG, N_LONG * FR_PER_TOK, "dual", rate=rate)
    arrive = ft[PKT12 - 1 :: PKT12]
    _, stall = schedule(arrive, np.full(len(arrive), PKT12 * 80.0))
    print(f"text at {num(rate, 1).removesuffix('.0'):>4s} tokens/s: first packet after {num(arrive[0], 0):>4s} ms, "
          f"paused {num(stall / 1000, 1)} s in total over {num(len(ft) / FPS12, 0)} s of speech")  # fmt: skip

From 12.5 tokens per second up, the speech never pauses; below that it does, because while the text is still
arriving, the released code's rule reads exactly one text token per frame. The slower the upstream model writes, the
later the first packet comes, too.

### A smaller RVQ
With 8 codes per layer instead of 16, does layer 3 still depend much on layer 2?

In [ ]:
CODES_K8 = rvq_codes(FR, k=8)
h1, h12 = cross_bits(CODES_K8, 2, [0], k=8), cross_bits(CODES_K8, 2, [0, 1], k=8)
print(f"8 codes per layer (at most 3 bits): layer 3 given layer 1 {num(h1, 3)} bits, given layers 1 and 2 "
      f"{num(h12, 3)} bits (saves {num(h1 - h12, 3)}; with 16 codes: saves {num(gap, 3)})")  # fmt: skip
assert h1 - h12 > 0.1

With 8 codes, the gap is smaller but still there: the dependence between the layers within a frame is not specific
to one configuration.

### A smaller update step
With noise σ = 0.03 in the logits instead of 0.1, does the product of the ratios over 2,000 tokens still spread
widely?

In [ ]:
LOGW3, KL3 = toy_ratios(0.03)
prod, gspo, tok, out = bands(LOGW3, 2000)
print(f"σ = 0.03 (KL per token {sci(KL3)} nat), L = 2,000: product {short(prod[0])} … {short(prod[2])} "
      f"(spans {num(math.log10(prod[2] / prod[0]), 1)} orders) · GSPO s {num(gspo[0], 4)} … {num(gspo[2], 4)} · "
      f"tokens outside the clipping range {pct(out, 2)}")  # fmt: skip

With a smaller update step, the product narrows but still spans almost two orders of magnitude, while almost no
token reaches GRPO's clipping range.

## 11 · Limitations

- **No large model runs here.** The Qwen3 language model, the Qwen2-Audio encoder, the DiT, BigVGAN, WavLM, the
  speaker encoder and the real MTP: we only rebuild the computations around them, with the numbers from the report
  and from the released code.
- **Section 4's per-layer measurements are Topic 05's**, on a small tokenizer (32 layers × 1,024 codes, ASR-style
  semantic supervision), not Qwen's 12 Hz tokenizer.
- **Section 6's two readings are the notebook's:** the 12 Hz version's RTF measured per frame, and whether the
  language model and the decoder run one after the other or overlap. The report makes neither clear.
- **Section 7:** the released code takes the whole text at once. Text arriving gradually, the upstream model's
  speed, one token per word and the time per frame are the notebook's assumptions.
- **Section 8** measures bits on a small RVQ of a mel spectrogram, not audio quality. **Section 9** is a toy with
  independent positions; it shows the spread of the ratios, not which algorithm trains better.
- **What the report does not publish**, the notebook does not have either: the data mix, the sizes of the tokenizer
  and of the speaker encoder, GSPO's reward rules, the number of DPO pairs, the length of each training stage.

## Summary

- **Two tokenizers:** 25 Hz × 1 codebook of 32,768 codes = 375 bits/s, and the main model runs 25 steps per second
  of speech; 12.5 Hz × 16 layers × 2,048 codes = 2,200 bits/s, and the main model runs only 12.5 steps per second,
  with MTP generating the remaining 187.5 tokens. 10 minutes of speech is 15,000 or 7,500 positions; stage S3
  extends the sequence from 8,192 to 32,768.
- **The first layer carries the content** (Topic 05's measurements): one layer already gives 81.9% correct digits
  but only 72.0% correct voices; from 1 to 4 layers, the voice gains 18.6 points, the digit 7.0. Without semantic
  supervision, the first layer drops to 71.9%.
- **Block-wise 25 Hz decoding:** blocks of 8 tokens, 3 blocks of look-back and 1 of look-ahead; the model waits for
  16 tokens, the first packet holds only 190 ms of audio, and the lead is always 450 ms. The list on p. 3 is 5
  blocks, not 4. The 12 Hz tokenizer needs no future context.
- **Latency:** all 12 rows of Table 2 are TTFP + decoding; 93 + 4 = 97 ms for the 12 Hz 0.6 B version. Five of the
  six 12 Hz RTFs match (TPP + decoding)/80 ms, as if TPP were the time for one frame rather than one packet (the
  notebook's reading). Playback never waits in steady state; only the 25 Hz version at 6 requests may pause for
  42 ms after the first packet, if the two stages run one after the other.
- **Two streams:** each step takes the sum of the previous frame's 16 embeddings plus the next text token (then
  padding once the text runs out); frame 0 needs only one text token, so the first packet does not depend on the
  length of the answer, as long as the text arrives no slower than 12.5 tokens per second.
- **MTP:** given layer 2, layer 3 saves 0.402 bits (3.884 → 3.483), which a parallel predictor misses; frame 0 has
  all 16 tokens after 1 large step, versus 16 for the delay pattern.
- **GSPO:** over 2,000 tokens, the product of the token ratios spans 6.4 orders of magnitude, while GSPO's geometric
  mean stays within 0.9916–0.9990.

## Review questions

1. A 6-second sentence: how many steps do the main models of the 25 Hz version and of the 12 Hz version run? How
   many small steps does MTP run?
2. If the 25 Hz decoder's blocks were 4 tokens and it still looked 1 block ahead, how many tokens would the model
   have to generate before the first packet, and how many ms of audio would that packet hold?
3. Why does the 25 Hz version's first-packet latency grow faster than the 12 Hz version's when the number of
   concurrent requests goes from 1 to 6?
4. In two-stream mode, what does the text stream hold once the text has run out? Why can't sequential mode start
   speaking while the text is still arriving?
5. What information does a parallel predictor for the 15 residual layers lack? Which computation does section 8 use
   to measure what it lacks?
6. Why can't the product of the ratios of 2,000 tokens be used as a weight, and what does GSPO do instead?

<details><summary>Answer hints</summary>

1. 6 · 25 = 150 steps for the 25 Hz version; 6 · 12.5 = 75 large steps for the 12 Hz version, plus
   75 · 15 = 1,125 small MTP steps.
2. Block 0 needs block 1, so the model waits for 8 tokens; a 4-token block is 160 ms of mel spectrogram, minus the
   130 ms that BigVGAN holds back, so the first packet holds only 30 ms of audio (section 10).
3. The 25 Hz version's decoding part (DiT, then BigVGAN) grows from 25 to 147 ms, while the 12 Hz version's causal
   ConvNet only goes from 4 to 5 ms; the language model's part grows for both.
4. The padding embedding `tts_pad` (after one "end of text" step, `tts_eos`). In sequential mode, frame 0 is
   predicted only after the model has read every text token, so it has to wait for the text to finish.
5. It does not know which codes the previous layers chose in the same frame, while in RVQ each layer quantizes the
   residual left after those layers. Section 8 compares layer 3's cross-entropy given layer 1 with that given layers
   1 and 2: the gap is the number of bits a parallel predictor misses.
6. The log of the product is a sum of 2,000 noisy numbers, so its spread grows with the square root of the length,
   and the product spans 6.4 orders of magnitude: a few sequences carry nearly all of the weight. GSPO uses the
   geometric mean (the |y|-th root of the product), which narrows as the sequence grows, and clips per response.
</details>

## Further reading

- Qwen Team (2026), *Qwen3-TTS Technical Report*: [arXiv:2601.15621](https://arxiv.org/abs/2601.15621); code and
  models: [QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS).
- Chu et al. (2024), *Qwen2-Audio Technical Report*: [arXiv:2407.10759](https://arxiv.org/abs/2407.10759).
- Défossez et al. (2024), *Moshi* (the Mimi tokenizer): [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Chen et al. (2022), *WavLM*: [arXiv:2110.13900](https://arxiv.org/abs/2110.13900).
- Lipman et al. (2023), *Flow Matching for Generative Modeling*: [arXiv:2210.02747](https://arxiv.org/abs/2210.02747);
  Peebles & Xie (2023), *DiT*: [arXiv:2212.09748](https://arxiv.org/abs/2212.09748); Lee et al. (2023), *BigVGAN*:
  [arXiv:2206.04658](https://arxiv.org/abs/2206.04658).
- Rafailov et al. (2023), *DPO*: [arXiv:2305.18290](https://arxiv.org/abs/2305.18290); Shao et al. (2024),
  *DeepSeekMath* (GRPO): [arXiv:2402.03300](https://arxiv.org/abs/2402.03300); Zheng et al. (2025), *GSPO*:
  [arXiv:2507.18071](https://arxiv.org/abs/2507.18071).
- Alibaba Token Foundry (2026), *Qwen-Audio-3.0-TTS*, another Alibaba TTS system, which supports Vietnamese:
  [arXiv:2607.23938](https://arxiv.org/abs/2607.23938).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Back to:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) ·
[Topic 05 · Audio tokenization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-hoa-am-thanh/) ·
[Topic 07 · MOSS-TTS](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/moss-tts/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)